# AI-Powered Customer Churn Analysis
## Jupyter Notebook — End-to-End Walkthrough

This notebook mirrors the Streamlit app pipeline for reproducibility and presentation.

**Steps covered:**
1. Data Loading & Inspection
2. Data Cleaning Pipeline
3. Exploratory Data Analysis
4. Business KPIs & SQL Analysis
5. Feature Engineering & Preprocessing
6. Model Training & Comparison
7. Model Evaluation
8. Churn Risk Predictions
9. AI-Powered Insights


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))  # make src/ importable

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.io as pio
pio.renderers.default = 'notebook'

from dotenv import load_dotenv
load_dotenv('../.env')

from src.data_cleaning import load_raw_data, clean_data, audit_dataframe, outlier_report
from src.eda import (
    fig_churn_distribution, fig_tenure_distribution,
    fig_monthly_charges_distribution, fig_contract_distribution,
    fig_correlation_heatmap, fig_churn_by_charges_bin,
)
from src.preprocessing import prepare_data
from src.model_training import train_all_models, select_best_model, detailed_evaluation
from src.prediction import predict_churn, add_risk_explanations, prediction_summary
from src.ai_insights import generate_insights
from src.utils import calculate_kpis, SQL_QUERIES, generate_sample_dataset

print('All imports OK.')

## 1. Data Loading

In [ ]:
# Try loading the real IBM Telco dataset; fall back to synthetic demo data
DATA_PATH = '../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'

if os.path.exists(DATA_PATH):
    raw_df = load_raw_data(DATA_PATH)
    print(f'Loaded real dataset: {raw_df.shape}')
else:
    print('Real dataset not found — generating synthetic demo data (n=1000).')
    print('Download the IBM Telco dataset from:')
    print('  https://www.kaggle.com/datasets/blastchar/telco-customer-churn')
    print('  and save it to data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
    raw_df = generate_sample_dataset(n=1000)

raw_df.head()

In [ ]:
# Raw data audit
audit_before = audit_dataframe(raw_df)
print(f"Shape         : {audit_before['n_rows']} rows × {audit_before['n_cols']} columns")
print(f"Total nulls   : {audit_before['total_nulls']}")
print(f"Duplicate rows: {audit_before['duplicate_rows']}")
print("\nData types:")
for col, dtype in audit_before['dtypes'].items():
    print(f"  {col:<30} {dtype}")

## 2. Data Cleaning

In [ ]:
clean_df, report = clean_data(raw_df)

print('=== CLEANING SUMMARY ===')
print(f"Rows     : {report['rows_before']} → {report['rows_after']}")
print(f"Nulls    : {report['nulls_before']} → {report['nulls_after']}")
print(f"Dupes rm : {report['duplicates_removed']}")
print(f"Churn encoded: {report['churn_encoded']}")
print()
print('Numeric conversions:')
for k, v in report['numeric_conversions'].items():
    print(f'  {k}: {v}')
print()
print('Missing value strategies:')
for k, v in report['missing_value_strategy'].items():
    print(f'  {k}: {v}')

In [ ]:
# Outlier report (non-destructive — for display only)
outlier_report(clean_df)

## 3. Exploratory Data Analysis

In [ ]:
fig_churn_distribution(clean_df).show()

In [ ]:
fig_tenure_distribution(clean_df).show()

In [ ]:
fig_monthly_charges_distribution(clean_df).show()

In [ ]:
fig_contract_distribution(clean_df).show()

In [ ]:
fig_churn_by_charges_bin(clean_df).show()

In [ ]:
fig_correlation_heatmap(clean_df).show()

## 4. Business KPIs

In [ ]:
kpis = calculate_kpis(clean_df)
for k, v in kpis.items():
    print(f'{k:<35} {v}')

## 5. SQL-Style Analysis (via Pandas)

In [ ]:
# Churn by contract type
print('--- Churn by Contract Type ---')
g = clean_df.groupby('Contract').agg(
    total=('Churn', 'count'),
    churned=('Churn', 'sum')
).reset_index()
g['churn_rate_%'] = (g['churned'] / g['total'] * 100).round(2)
print(g.sort_values('churn_rate_%', ascending=False).to_string(index=False))

In [ ]:
# Revenue from churned customers
if 'TotalCharges' in clean_df.columns:
    churned = clean_df[clean_df['Churn'] == 1]
    print(f'Total revenue from churned customers : ${churned["TotalCharges"].sum():,.2f}')
    print(f'Average revenue per churned customer : ${churned["TotalCharges"].mean():,.2f}')

## 6. Feature Engineering & Preprocessing

In [ ]:
prep = prepare_data(clean_df)
print(f'Training samples : {prep["X_train"].shape[0]}')
print(f'Test samples     : {prep["X_test"].shape[0]}')
print(f'Features         : {prep["X_train"].shape[1]}')
print(f'\nChurn rate train : {prep["y_train"].mean()*100:.2f}%')
print(f'Churn rate test  : {prep["y_test"].mean()*100:.2f}%')

## 7. Model Training & Comparison

In [ ]:
results_df, trained = train_all_models(
    prep['X_train'], prep['X_test'],
    prep['y_train'], prep['y_test'],
)
results_df

In [ ]:
from src.eda import fig_model_comparison
fig_model_comparison(results_df).show()

## 8. Model Evaluation

In [ ]:
best_name, best_model = select_best_model(results_df, trained)
print(f'Best model: {best_name}')

eval_info = detailed_evaluation(
    best_model,
    prep['X_test'],
    prep['y_test'],
    feature_names=prep['feature_names_transformed'],
)
print(f'ROC-AUC: {eval_info["auc"]}')
print()
print(eval_info['report_text'])

In [ ]:
from src.eda import fig_confusion_matrix, fig_roc_curve, fig_feature_importance

fig_confusion_matrix(eval_info['confusion_matrix']).show()
fig_roc_curve(eval_info['fpr'], eval_info['tpr'], eval_info['auc']).show()

if eval_info['feature_importances'] is not None:
    fig_feature_importance(eval_info['feature_names'], eval_info['feature_importances']).show()

## 9. Churn Risk Predictions

In [ ]:
pred_df = predict_churn(
    clean_df, best_model,
    prep['preprocessor'],
    prep['feature_cols'],
)
pred_df = add_risk_explanations(pred_df)

summary = prediction_summary(pred_df)
for k, v in summary.items():
    print(f'{k:<30} {v}')

In [ ]:
# View high-risk customers
id_col = next((c for c in pred_df.columns if 'id' in c.lower()), None)
display_cols = ([id_col] if id_col else []) + [
    c for c in ['Contract', 'tenure', 'MonthlyCharges',
                'InternetService', 'Churn Probability', 'Risk Level', 'Risk Factors']
    if c in pred_df.columns
]
high_risk = pred_df[pred_df['Risk Level'] == 'High Risk'][display_cols]
print(f'High-risk customers: {len(high_risk)}')
high_risk.sort_values('Churn Probability', ascending=False).head(10)

## 10. AI-Powered Insights

In [ ]:
insights = generate_insights(clean_df, pred_df)
print(f"Insight mode: {insights['mode']}")
print()
print(insights['key_findings'])

In [ ]:
print(insights['churn_drivers'])

In [ ]:
print(insights['recommendations'])

In [ ]:
# Save outputs
os.makedirs('../outputs', exist_ok=True)
pred_df.to_csv('../outputs/predictions.csv', index=False)
high_risk.to_csv('../outputs/high_risk_customers.csv', index=False)
print('Outputs saved.')

---
## Summary

This notebook demonstrated:
- Automated data cleaning with a full audit trail
- Exploratory Data Analysis with Plotly visualisations
- Business KPIs and SQL-equivalent analysis
- Training and comparing 4 ML models
- Detailed model evaluation (ROC-AUC, confusion matrix, classification report)
- Per-customer churn risk scoring and explanations
- AI/rule-based insight generation

For the interactive dashboard, run:
```bash
streamlit run app.py
```
